In [0]:
%run ../00_Setup/01_Config

In [0]:
%run ../Utils/Common_Utils

In [0]:
# Metadata
PIPELINE_NAME = "Silver_Payment_Load"
RUN_ID = generate_run_id()
START_TIME = start_pipeline()

Pipeline Started : 2026-07-18 12:32:11.903191


In [0]:
SOURCE_TABLE = TARGET_TABLE_PAYMENTS
TARGET_TABLE = SILVER_PAYMENTS
QUARANTINE_TABLE = QUARANTINE_PAYMENTS

In [0]:
# Read Bronze Payment Items
payments_df = spark.table(SOURCE_TABLE)

In [0]:
total_rows = payments_df.count()

In [0]:
# Dataset Profile
print("PAYMENTS DATASET PROFILE")
print(f"Total Records : {total_rows}")
payments_df.printSchema()
display(payments_df.limit(10))

PAYMENTS DATASET PROFILE
Total Records : 57388
root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_date: date (nullable = true)
 |-- pipeline_name: string (nullable = true)
 |-- run_id: string (nullable = true)



order_id,payment_sequential,payment_type,payment_installments,payment_value,ingestion_timestamp,ingestion_date,pipeline_name,run_id
ORD_0000001,1,voucher,12,2775.42,2026-07-15T06:01:54.899Z,2026-07-15,Bronze_Payments_Load,3532582a-9b9d-4aa6-9775-be38ea50d344
ORD_0000001,2,credit_card,1,32.43,2026-07-15T06:01:54.899Z,2026-07-15,Bronze_Payments_Load,3532582a-9b9d-4aa6-9775-be38ea50d344
ORD_0000002,1,voucher,1,805.73,2026-07-15T06:01:54.899Z,2026-07-15,Bronze_Payments_Load,3532582a-9b9d-4aa6-9775-be38ea50d344
ORD_0000003,1,credit_card,6,508.88,2026-07-15T06:01:54.899Z,2026-07-15,Bronze_Payments_Load,3532582a-9b9d-4aa6-9775-be38ea50d344
ORD_0000004,1,boleto,2,1589.64,2026-07-15T06:01:54.899Z,2026-07-15,Bronze_Payments_Load,3532582a-9b9d-4aa6-9775-be38ea50d344
ORD_0000004,2,boleto,3,35.07,2026-07-15T06:01:54.899Z,2026-07-15,Bronze_Payments_Load,3532582a-9b9d-4aa6-9775-be38ea50d344
ORD_0000005,1,credit_card,3,1592.91,2026-07-15T06:01:54.899Z,2026-07-15,Bronze_Payments_Load,3532582a-9b9d-4aa6-9775-be38ea50d344
ORD_0000006,1,credit_card,1,495.97,2026-07-15T06:01:54.899Z,2026-07-15,Bronze_Payments_Load,3532582a-9b9d-4aa6-9775-be38ea50d344
ORD_0000007,1,credit_card,2,1738.92,2026-07-15T06:01:54.899Z,2026-07-15,Bronze_Payments_Load,3532582a-9b9d-4aa6-9775-be38ea50d344
ORD_0000008,1,voucher,1,1107.69,2026-07-15T06:01:54.899Z,2026-07-15,Bronze_Payments_Load,3532582a-9b9d-4aa6-9775-be38ea50d344


In [0]:
# Data Quality Checks
distinct_keys = payments_df.select("order_id", "payment_sequential").distinct().count()
primary_key_status = (distinct_keys == total_rows)

print(f"Distinct Composite Keys : {distinct_keys}")
print(f"Primary Key Validation  : {primary_key_status}")

duplicate_rows = duplicate_summary(payments_df, total_rows)
null_summary(payments_df)

Distinct Composite Keys : 57388
Primary Key Validation  : True
Duplicate Rows : 0


order_id,payment_sequential,payment_type,payment_installments,payment_value,ingestion_timestamp,ingestion_date,pipeline_name,run_id
0,0,0,0,0,0,0,0,0


In [0]:
# Data Standardization
payments_df = payments_df.withColumn(
    "payment_type",
    F.trim(F.lower(F.col("payment_type")))
)

In [0]:
invalid_business_rules = (
    payments_df
    .filter(
        (F.col("payment_value") <= 0) |
        (F.col("payment_installments") < 1) |
        (F.col("payment_sequential") < 1) |
        (F.col("payment_type").isNull()) |
        (F.trim(F.col("payment_type")) == "") |
        (~F.col("payment_type").isin(
            "credit_card",
            "boleto",
            "voucher",
            "debit_card",
            "not_defined"
        ))
    )
    .withColumn(
        "quarantine_reason",
        F.when(F.col("payment_value") <= 0,
               "Invalid payment value")
        .when(F.col("payment_installments") < 1,
               "Invalid payment installments")
        .when(F.col("payment_sequential") < 1,
               "Invalid payment sequence")
        .when(F.col("payment_type").isNull(),
               "Payment type is NULL")
        .when(F.trim(F.col("payment_type")) == "",
               "Payment type is blank")
        .otherwise("Invalid payment type")
    ).withColumn("validation_type", F.lit("Business Rule"))
)
business_rule_count = invalid_business_rules.count()
print(f"Business Rule Violations : {business_rule_count}")

Business Rule Violations : 0


In [0]:
# Foreign Key Validation
silver_orders = spark.table(SILVER_ORDERS)

orphan_payments = (
    payments_df.join(
        silver_orders.select("order_id"),
        on="order_id",
        how="left_anti"
    )
    .withColumn("quarantine_reason", F.lit("Order ID Not Found in Silver Orders"))
    .withColumn("validation_type", F.lit("Foreign Key"))
)

orphan_count = orphan_payments.count()
print(f"Orphan Payments : {orphan_count}")

Orphan Payments : 0


In [0]:
# Invalid Payments (combined)
invalid_payments = (
    invalid_business_rules
    .unionByName(orphan_payments)
    .dropDuplicates(["order_id", "payment_sequential"])
)
print(f"Total Invalid Payments : {invalid_payments.count()}")

Total Invalid Payments : 0


In [0]:
# Valid Payments
valid_payments = (
    payments_df.join(
        invalid_payments.select("order_id", "payment_sequential"),
        on=["order_id", "payment_sequential"],
        how="left_anti"
    )
)
print(f"Valid Payments : {valid_payments.count()}")

Valid Payments : 57388


In [0]:
valid_payments = add_audit_columns(
    valid_payments,
    PIPELINE_NAME,
    RUN_ID
)

In [0]:
# Qurantine Write
(
    invalid_payments.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)
print(f"Quarantined Records : {invalid_payments.count()}")

Quarantined Records : 0


In [0]:
from delta.tables import DeltaTable

if not spark.catalog.tableExists(TARGET_TABLE):
    (
        valid_payments.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(TARGET_TABLE)
    )
else:
    target = DeltaTable.forName(spark, TARGET_TABLE)
    (
        target.alias("target")
        .merge(
            valid_payments.alias("source"),
            """
            target.order_id = source.order_id
            AND
            target.payment_sequential = source.payment_sequential
            """
        )
        .whenMatchedUpdate(
            condition="""
                coalesce(target.payment_type,'')
                <>
                coalesce(source.payment_type,'')

                OR

                coalesce(target.payment_installments,-1)
                <>
                coalesce(source.payment_installments,-1)

                OR

                coalesce(target.payment_value,-1)
                <>
                coalesce(source.payment_value,-1)

            """,

            set={

                "payment_type":
                    "source.payment_type",

                "payment_installments":
                    "source.payment_installments",

                "payment_value":
                    "source.payment_value",

                "ingestion_timestamp":
                    "source.ingestion_timestamp",

                "ingestion_date":
                    "source.ingestion_date",

                "pipeline_name":
                    "source.pipeline_name",

                "run_id":
                    "source.run_id"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )
print("Silver Payments loaded successfully.")

Silver Payments loaded successfully.


In [0]:
print("SILVER PAYMENTS VALIDATION SUMMARY")
print(f"Total Source Records      : {total_rows}")
print(f"Duplicate Records         : {duplicate_rows}")
print(f"Invalid Records           : {invalid_payments.count()}")
print(f"Valid Records             : {valid_payments.count()}")
print(f"Quarantined Records       : {invalid_payments.count()}")


print("Silver Payments Pipeline Completed Successfully")

SILVER PAYMENTS VALIDATION SUMMARY
Total Source Records      : 57388
Duplicate Records         : 0
Invalid Records           : 0
Valid Records             : 57388
Quarantined Records       : 0
Silver Payments Pipeline Completed Successfully


In [0]:
try:
    rows_written = valid_payments.count()
    execution_status = "SUCCESS"

except Exception as e:
    rows_written = 0
    execution_status = "FAILED"
    raise

In [0]:
bronze_load_report(
    pipeline_name=PIPELINE_NAME,
    run_id=RUN_ID,
    source=SOURCE_TABLE,
    target=TARGET_TABLE,
    rows_read=total_rows,
    rows_written=rows_written,
    duplicate_count=duplicate_rows,
    start_time=START_TIME,
    status=execution_status
)

LOAD REPORT
Pipeline        : Silver_Payment_Load
Run ID          : 8092b74a-e10a-4094-8439-aac43d193fd9
Source          : retailmart.bronze.payments
Target          : retailmart.silver.payments
Rows Read       : 57388
Rows Written    : 57388
Duplicate Rows  : 0
Start Time      : 2026-07-18 12:32:11.903191
End Time        : 2026-07-18 12:56:28.527810
Duration (sec)  : 1456.62
Status          : SUCCESS


# Engineering Observations

- Composite primary key (order_id, payment_sequential) validated successfully.
- Duplicate and null value analysis completed.
- Standardized payment_type using trim() and lowercase().
- Applied business rule validations for payment value, installments, sequence number, and payment type.
- Validated foreign key relationship with Silver Orders.
- Invalid records were quarantined before loading.
- Audit metadata refreshed before loading into the Silver layer.
- Implemented incremental loading using Delta Lake SCD Type 1 MERGE.
- Conditional updates reduce unnecessary writes by updating only changed records.